# 대리 모델 실습

**Surrogate Model**

비싼 실험이나 계산의 결과를 빠르게 근사하는 모델.

소재 분야에서 이해하기: DFT를 매번 수행하는 대신 물성 예측 모델로 후보를 좁힌다.

이 노트북은 개념을 직접 돌려보기 위한 예제입니다. 데이터는 실제 측정값이 아니라 개념 확인용으로
생성한 값이므로, 결과 수치를 연구 결론으로 쓰지 마세요. 위에서부터 순서대로 실행하세요.
그림의 축 이름은 기본 폰트에 한글 글리프가 없어 영문으로 적었습니다.

참고 자료: [베이지안 능동학습 연구](https://www.nature.com/articles/s41467-020-19597-w)

## 1. 비싼 계산을 값싼 모델로 대신하기

In [ ]:
import numpy as np
import matplotlib.pyplot as plt

rng = np.random.default_rng(0)
plt.rcParams['figure.figsize'] = (7, 4)

import time

def expensive_simulation(x):
    """비싼 계산을 흉내냅니다(의도적으로 느리게)."""
    time.sleep(0.002)
    return np.sin(3 * x[0]) + 0.5 * x[1] ** 2 - 0.3 * x[0] * x[1]

start = time.perf_counter()
samples = rng.random((120, 2))
values = np.array([expensive_simulation(point) for point in samples])
elapsed = time.perf_counter() - start
print('참조 계산 %d회에 %.2f초 (1회 %.4f초)' % (len(samples), elapsed, elapsed / len(samples)))

In [ ]:
from sklearn.gaussian_process import GaussianProcessRegressor
from sklearn.gaussian_process.kernels import RBF, ConstantKernel
from sklearn.metrics import mean_absolute_error

surrogate = GaussianProcessRegressor(kernel=ConstantKernel(1.0) * RBF([0.4, 0.4]),
                                     normalize_y=True, random_state=0).fit(samples, values)

test_points = rng.random((300, 2))
truth = np.array([np.sin(3 * p[0]) + 0.5 * p[1] ** 2 - 0.3 * p[0] * p[1] for p in test_points])
start = time.perf_counter()
predicted = surrogate.predict(test_points)
surrogate_time = time.perf_counter() - start
print('대리 모델 %d회 예측에 %.4f초' % (len(test_points), surrogate_time))
print('예측 MAE %.4f (목표값 표준편차 %.4f)' % (mean_absolute_error(truth, predicted), truth.std()))
print('속도 이득 약 %.0f배' % ((elapsed / len(samples)) / (surrogate_time / len(test_points))))

## 2. 학습 데이터 수에 따른 신뢰도

In [ ]:
for n in (10, 30, 60, 120):
    model = GaussianProcessRegressor(kernel=ConstantKernel(1.0) * RBF([0.4, 0.4]),
                                     normalize_y=True, random_state=0).fit(samples[:n], values[:n])
    mean, std = model.predict(test_points, return_std=True)
    print('참조 %3d회 학습 -> MAE %.4f, 평균 예측 표준편차 %.4f'
          % (n, mean_absolute_error(truth, mean), std.mean()))
print('\n대리 모델은 참조 계산을 대체하는 것이 아니라 후보를 좁히는 도구입니다.')
print('최종 후보는 반드시 참조 계산이나 실험으로 다시 확인해야 합니다.')

---

셀의 숫자를 바꿔가며 다시 실행해보면 개념이 더 분명해집니다. 용어 사전으로 돌아가려면
[소재·AI 용어 사전](https://forum.rnddata.org/glossary/#surrogate)을 여세요.